# 02 Ablation Study
**Paper**: XDenseQNet: Hybrid Quantum Neural Network for Skin Lesion Classification
**Authors**: Shehroz Tariq, Umair Inayat, Fatima Zulfiqar, Rehan Raza*, Muhammad Arif
**Description**: Component-wise ablation: A1 (no QNN), A2 (classical ML classifiers on frozen features), A3 (no CBAM and no QNN). Compares against the full XDenseQNet pipeline.
**Runtime**: ~2-3 hours on a single GPU


In [ ]:
# !pip install -r ../requirements.txt


In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Ablation Study — DenseNet121 + CBAM + Hybrid QNN (4Q-2L)
## MSLD v2.0 · 4-Class · Original Training Dataset & Augmentation

### Ablations Performed

| # | Config | Description |
|---|--------|-------------|
| **A1** | DenseNet121 + CBAM + DenseHead + Softmax | Remove QNN → keep exact backbone+CBAM+deep classifier head, add Softmax |
| **A2** | DenseNet121 + CBAM → Classical ML | Remove QNN → extract features with frozen DenseNet121+CBAM, classify with SVM/RF/LR/KNN/GBT |
| **A3** | 8 Backbones × Standard Classifier + Softmax | Remove QNN + CBAM + custom head → each backbone's own linear head + Softmax |

**Same dataset & augmentation as original training:**
- Original dataset: `Monkeypox Skin Image Dataset`
- Balanced augmentation to 550 samples/class for training
- Splits: 70% train / 15% val / 15% test  
- Transforms: HFlip, VFlip, Rotate, ShiftScaleRotate, BrightnessContrast, HSV, GaussNoise/Blur, CoarseDropout
- Val/Test: Resize + Normalize only (no augmentation)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
import subprocess, sys
for pkg in ["pennylane","albumentations","scikit-learn","timm","seaborn","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)
print("✓ All packages installed")


In [ ]:
import os, time, warnings, copy, json, logging, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import timm
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score,
                              roc_curve, auc)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

print("✓ All imports successful")
print(f"  PyTorch : {torch.__version__}")
print(f"  Device  : {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## ⚙️ Configuration — Mirrors original training exactly

In [ ]:
# ============================================================
#   DATASET PATHS — set ORIGINAL_DATASET_PATH to your raw dataset location
# ============================================================
#
#  The original training notebook used:
#    original_dataset_path = "Monkeypox Skin Image Dataset"
#
#  This is a folder that contains 4 subfolders:
#    Normal/  (or any folder whose name contains "normal")
#    Monkeypox/
#    Chickenpox/
#    Measles/
#
#  Common Kaggle paths (uncomment the one that applies):
#
#  Option A — Kaggle dataset "mpox-skin-lesion-dataset"
# ORIGINAL_DATASET_PATH = "/kaggle/input/mpox-skin-lesion-dataset/Monkeypox Skin Image Dataset"
#
#  Option B — dataset uploaded/extracted manually in Colab / local
# ORIGINAL_DATASET_PATH = "Monkeypox Skin Image Dataset"
#
#  Option C — Kaggle MSLD v2 (Original Images flat folder, not FOLDS)
# ORIGINAL_DATASET_PATH = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images"
#
#  !! The folder you point to must contain class subfolders directly,
#     e.g.  ORIGINAL_DATASET_PATH/Normal/, ORIGINAL_DATASET_PATH/Monkeypox/, ...
#
# ─────────────────────────────────────────────────────────────────────────────

import os

# Auto-detect common Kaggle paths for this dataset
_candidate_paths = [
    # Kaggle MSLD v1 dataset
    # "/kaggle/input/mpox-skin-lesion-dataset/Monkeypox Skin Image Dataset",
    # "/kaggle/input/monkeypox-skin-lesion-dataset/Monkeypox Skin Image Dataset",
    # # Kaggle MSLD v2 original images (flat, not FOLDS)
    # "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images",
    # # Colab / local relative path (same as training notebook)
    "/content/drive/MyDrive/Monkey Pox/Monkeypox Skin Image Dataset",
]

ORIGINAL_DATASET_PATH = None
for _p in _candidate_paths:
    if os.path.isdir(_p):
        # Verify at least one class subfolder exists
        _subdirs = [d for d in os.listdir(_p) if os.path.isdir(os.path.join(_p, d))]
        _class_matches = [d for d in _subdirs
                          if any(cls.lower() in d.lower()
                                 for cls in ["normal","monkeypox","chickenpox","measles","healthy"])]
        if len(_class_matches) >= 2:
            ORIGINAL_DATASET_PATH = _p
            print(f"✓ Auto-detected dataset at: {_p}")
            print(f"  Subfolders found: {_subdirs}")
            break

if ORIGINAL_DATASET_PATH is None:
    # ── MANUAL OVERRIDE — set this if auto-detect fails ──────────────────────
    ORIGINAL_DATASET_PATH = "Monkeypox Skin Image Dataset"
    print(f"⚠ Could not auto-detect dataset. Using: {ORIGINAL_DATASET_PATH}")
    print("  If this path is wrong, edit ORIGINAL_DATASET_PATH manually above.")

# Where the balanced processed dataset will be written (mirrors training notebook)
PROCESSED_DATASET_PATH = "processed_balanced_dataset"

# Output directory for ablation results
OUTPUT_DIR = "ablation_study_results"

# ============================================================
# All settings below EXACTLY mirror Config class from training
# ============================================================

DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE      = (224, 224)
BATCH_SIZE      = 16          # same as training (config.batch_size)
NUM_CLASSES     = 4
CLASS_NAMES     = ["Normal", "Monkeypox", "Chickenpox", "Measles"]
# subfolders expected: normal/, monkeypox/, chickenpox/, measles/ (lower-case)
CLASS_DIRS      = ["normal", "monkeypox", "chickenpox", "measles"]

TARGET_SAMPLES  = 550         # config.target_samples_per_class
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.15
TEST_RATIO      = 0.15
DROPOUT         = 0.5         # config.dropout_rate
WEIGHT_DECAY    = 1e-3        # config.weight_decay

# Training hyper-params (mirror original Phase 1 / Phase 2)
PRETRAIN_EPOCHS   = 30        # config.pretrain_epochs
PRETRAIN_LR       = 0.001     # config.pretrain_lr
NUM_EPOCHS        = 80        # config.num_epochs  (A1/A3)
PATIENCE          = 20        # config.patience
LEARNING_RATE     = 0.0008    # config.learning_rate
UNFREEZE_EPOCH    = 15        # config.unfreeze_epoch
COSINE_T0         = 10        # config.cosine_T_0
COSINE_TMULT      = 2         # config.cosine_T_mult
LABEL_SMOOTHING   = 0.1       # config.label_smoothing
FOCAL_GAMMA       = 2.0       # config.focal_gamma
USE_MIXUP         = True      # config.use_mixup
MIXUP_ALPHA       = 0.2       # config.mixup_alpha

BACKBONE_FEATURES = {         # config.backbones[key]["features"]
    "densenet121"     : 1024,

}
BACKBONE_TYPE = {

    "densenet121"     : "torchvision",

}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Logging ──────────────────────────────────────────────────────────────────
log_path = os.path.join(OUTPUT_DIR, "ablation_study.log")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    handlers=[logging.FileHandler(log_path, mode="w"),
              logging.StreamHandler()]
)
log = logging.getLogger("ablation")
log.info("Ablation Study initialised")
log.info(f"Dataset  : {ORIGINAL_DATASET_PATH}")
log.info(f"Balanced : {PROCESSED_DATASET_PATH}")
log.info(f"Output   : {OUTPUT_DIR}")
log.info(f"Classes  : {CLASS_NAMES}")
log.info(f"Device   : {DEVICE}")

print("✓ Config ready")
print(f"  Original dataset  : {ORIGINAL_DATASET_PATH}")
print(f"  Processed dataset : {PROCESSED_DATASET_PATH}")
print(f"  Output dir        : {OUTPUT_DIR}/")


## 🗂️ Dataset Organisation (identical to original training)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BalancedDataBalancer — copied verbatim from 11_backbones training notebook
# ─────────────────────────────────────────────────────────────────────────────
class BalancedDataBalancer:
    """Balanced augmentation - all classes to target (exact copy from training)"""
    def __init__(self):
        self.augmentation_transform = A.Compose([
            A.Resize(IMAGE_SIZE[0], IMAGE_SIZE[1]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=20, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=20, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
                A.GaussianBlur(blur_limit=(3, 5), p=0.5),
            ], p=0.3),
            A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                            min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    def generate_augmented_images(self, image_path, num_augmentations):
        try:
            image = Image.open(image_path).convert("RGB").resize(IMAGE_SIZE, Image.LANCZOS)
            image_np = np.array(image)
            augmented = []
            for _ in range(num_augmentations):
                aug = self.augmentation_transform(image=image_np)["image"]
                # De-normalise back to PIL for saving
                mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
                std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
                img_np = torch.clamp(aug*std+mean, 0, 1).permute(1,2,0).numpy()
                augmented.append(Image.fromarray((img_np*255).astype(np.uint8)))
            return augmented
        except:
            return []


# ─────────────────────────────────────────────────────────────────────────────
# get_transforms — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
def get_transforms():
    """Training: strong augmentation. Val/Test: resize + normalise only."""
    train_transform = A.Compose([
        A.Resize(IMAGE_SIZE[0], IMAGE_SIZE[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=15, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.GaussianBlur(blur_limit=(3, 5), p=0.5),
        ], p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                        min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    val_test_transform = A.Compose([
        A.Resize(IMAGE_SIZE[0], IMAGE_SIZE[1]),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    return train_transform, val_test_transform, val_test_transform


# ─────────────────────────────────────────────────────────────────────────────
# MultiClassDataset — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
class MultiClassDataset(Dataset):
    def __init__(self, data_dir, transform=None, split_name="train"):
        self.transform   = transform
        self.images      = []
        self.labels      = []
        self.class_counts= {}
        for class_idx, class_name in enumerate(CLASS_DIRS):
            class_path = os.path.join(data_dir, class_name)
            if os.path.exists(class_path):
                imgs = [os.path.join(class_path, f) for f in os.listdir(class_path)
                        if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".tiff"))]
                self.images.extend(imgs)
                self.labels.extend([class_idx]*len(imgs))
                self.class_counts[class_idx] = len(imgs)
        print(f"  {split_name.upper()}: {len(self.images)} images")
        for i, n in enumerate(CLASS_NAMES):
            print(f"    {n}: {self.class_counts.get(i,0)}")

    def get_class_weights(self):
        total = sum(self.class_counts.values())
        n     = NUM_CLASSES
        return torch.FloatTensor([total/(n*max(self.class_counts.get(i,1),1))
                                   for i in range(n)])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.images[idx]).convert("RGB")
            img = img.resize(IMAGE_SIZE, Image.LANCZOS)
            img = np.array(img)
        except:
            img = np.zeros((IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, self.labels[idx]


# ─────────────────────────────────────────────────────────────────────────────
# organize_balanced_dataset — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
def organize_balanced_dataset(source_path, target_path):
    """Organise with BALANCED augmentation — all classes to ~550 (exact copy)."""
    print("\nSETTING UP DATASET")
    print("="*70)
    if os.path.exists(target_path):
        shutil.rmtree(target_path)
    os.makedirs(target_path, exist_ok=True)

    for split in ["train","val","test"]:
        for cls in CLASS_NAMES:
            os.makedirs(os.path.join(target_path, split, cls.lower()), exist_ok=True)

    # Collect per-class images
    all_class_images = {}
    for class_name in CLASS_NAMES:
        for folder in os.listdir(source_path):
            fp = os.path.join(source_path, folder)
            if os.path.isdir(fp) and class_name.lower() in folder.lower():
                imgs = [os.path.join(fp, f) for f in os.listdir(fp)
                        if f.lower().endswith((".png",".jpg",".jpeg",".bmp",".tiff"))]
                all_class_images[class_name] = imgs
                print(f"  {class_name}: {len(imgs)} original images")
                break

    # Split and copy
    split_counts = {"train":{}, "val":{}, "test":{}}
    for class_name, images in all_class_images.items():
        random.shuffle(images)
        total       = len(images)
        train_size  = int(total * TRAIN_RATIO)
        val_size    = int(total * VAL_RATIO)
        splits_data = {
            "train": images[:train_size],
            "val"  : images[train_size:train_size+val_size],
            "test" : images[train_size+val_size:]
        }
        for split, split_images in splits_data.items():
            dst = os.path.join(target_path, split, class_name.lower())
            for i, img_path in enumerate(split_images):
                shutil.copy2(img_path,
                             os.path.join(dst, f"{class_name.lower()}_{split}_{i:04d}.jpg"))
            split_counts[split][class_name] = len(split_images)

    # Balanced augmentation for training
    print("\nAugmenting TRAINING data to", TARGET_SAMPLES, "per class...")
    balancer = BalancedDataBalancer()
    for class_name in CLASS_NAMES:
        current   = split_counts["train"][class_name]
        needed    = TARGET_SAMPLES - current
        target_dir= os.path.join(target_path, "train", class_name.lower())
        if needed > 0:
            orig_files = [f for f in os.listdir(target_dir)
                          if f.endswith((".jpg",".jpeg",".png"))][:15]
            total_gen  = 0
            per_image  = max(1, needed // len(orig_files))
            for i, img_file in enumerate(orig_files):
                if total_gen >= needed: break
                aug_imgs = balancer.generate_augmented_images(
                    os.path.join(target_dir, img_file), per_image)
                for j, aug_img in enumerate(aug_imgs):
                    if total_gen >= needed: break
                    aug_img.save(
                        os.path.join(target_dir,
                                     f"{class_name.lower()}_aug_{i:04d}_{j:02d}.jpg"),
                        "JPEG", quality=95)
                    total_gen += 1
            print(f"  {class_name}: {current} → {current+total_gen} (+{total_gen})")
        else:
            print(f"  {class_name}: {current} (already at target)")

    print("\nFinal split counts:")
    for split in ["train","val","test"]:
        tot = sum(split_counts[split].values())
        print(f"  {split.upper()} total: {tot}")
        for cls in CLASS_NAMES:
            print(f"    {cls}: {split_counts[split][cls]}")
    return True


def create_dataloaders():
    """Create data loaders (mirrors training notebook create_dataloaders)."""
    train_tf, val_tf, test_tf = get_transforms()
    train_ds = MultiClassDataset(os.path.join(PROCESSED_DATASET_PATH,"train"),
                                  train_tf, "train")
    val_ds   = MultiClassDataset(os.path.join(PROCESSED_DATASET_PATH,"val"),
                                  val_tf,   "val")
    test_ds  = MultiClassDataset(os.path.join(PROCESSED_DATASET_PATH,"test"),
                                  test_tf,  "test")
    kw = dict(num_workers=0, pin_memory=True)
    return {
        "train_loader": DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True, **kw),
        "val_loader"  : DataLoader(val_ds,   BATCH_SIZE, shuffle=False, **kw),
        "test_loader" : DataLoader(test_ds,  BATCH_SIZE, shuffle=False, **kw),
        "train_dataset": train_ds,
    }

print("✓ Dataset classes & pipeline ready")


In [ ]:
# ── Validate dataset path ──────────────────────────────────────────────────────
print(f"Using original dataset: {ORIGINAL_DATASET_PATH}")

if not os.path.isdir(ORIGINAL_DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found at: {ORIGINAL_DATASET_PATH}\n"
        "Please set ORIGINAL_DATASET_PATH in the config cell above to point to "
        "your 'Monkeypox Skin Image Dataset' folder.")

# Show what subfolders exist
_subdirs = sorted([d for d in os.listdir(ORIGINAL_DATASET_PATH)
                    if os.path.isdir(os.path.join(ORIGINAL_DATASET_PATH, d))])
print(f"Subfolders found in dataset root:")
for d in _subdirs:
    n = len([f for f in os.listdir(os.path.join(ORIGINAL_DATASET_PATH, d))
              if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".tiff"))])
    print(f"  {d}: {n} images")

# Verify all 4 classes can be matched
_required = ["Normal", "Monkeypox", "Chickenpox", "Measles"]
_missing = []
for cls in _required:
    matched = any(cls.lower() in d.lower() for d in _subdirs)
    if not matched:
        _missing.append(cls)
if _missing:
    print(f"\n⚠ WARNING: Could not find folders matching: {_missing}")
    print("  organize_balanced_dataset() will skip these classes.")
    print("  Check your dataset folder structure.")
else:
    print(f"\n✓ All 4 class folders detected: {_required}")

# ── Organise dataset (skip if already done) ────────────────────────────────
if not os.path.exists(os.path.join(PROCESSED_DATASET_PATH, "train")):
    print("\nOrganising dataset from scratch (balanced augmentation to 550/class)...")
    success = organize_balanced_dataset(ORIGINAL_DATASET_PATH, PROCESSED_DATASET_PATH)
    if not success:
        raise RuntimeError("Dataset organisation failed — check ORIGINAL_DATASET_PATH")
else:
    print(f"\n✓ Processed dataset already exists at '{PROCESSED_DATASET_PATH}' — skipping re-organisation")
    print("  (Delete this folder to force re-organisation)")

# ── Create data loaders shared by all ablations ───────────────────────────
print("\nCreating data loaders...")
data_loaders = create_dataloaders()
print("\n✓ Data loaders ready")
print(f"  Train batches : {len(data_loaders['train_loader'])}")
print(f"  Val   batches : {len(data_loaders['val_loader'])}")
print(f"  Test  batches : {len(data_loaders['test_loader'])}")


## 🧱 Shared Architecture (exact copy from training notebook)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CBAM — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        reduced = max(in_channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x): return x * self.sigmoid(self.fc(x))

class SpatialAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // 4),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // 4, in_channels),
            nn.Sigmoid()
        )
    def forward(self, x): return x * self.fc(x)

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(in_channels)
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ─────────────────────────────────────────────────────────────────────────────
# UniversalCNNExtractor — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
class UniversalCNNExtractor(nn.Module):
    """Universal feature extractor for all backbones (exact copy from training)."""
    def __init__(self, backbone_key):
        super().__init__()
        self.backbone_key      = backbone_key
        self.backbone_features = BACKBONE_FEATURES[backbone_key]
        self.backbone_type     = BACKBONE_TYPE[backbone_key]
        self.backbone          = self._load_backbone()
        self._configure_backbone()
        self.adaptive_pool     = nn.AdaptiveAvgPool2d((1,1))
        self.feature_normalizer= nn.Sequential(
            nn.BatchNorm1d(self.backbone_features), nn.ReLU()
        )
        self.temp_classifier   = nn.Linear(self.backbone_features, NUM_CLASSES)
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"  {backbone_key}: {self.backbone_features} features, {trainable:,} trainable params")

    def _load_backbone(self):
        key = self.backbone_key
        if key == "resnet18":
            m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            return nn.Sequential(*list(m.children())[:-1])
        elif key == "resnet50":
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
            return nn.Sequential(*list(m.children())[:-1])
        elif key == "densenet121":
            m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
            m.classifier = nn.Identity(); return m
        elif key == "efficientnet_b0":
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            m.classifier = nn.Identity(); return m
        elif key == "mobilenet_v2":
            m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
            m.classifier = nn.Identity(); return m
        elif key == "vgg16":
            m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
            m.classifier = nn.Sequential(*list(m.classifier.children())[:-1]); return m
        elif key == "convnext_tiny":
            return timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        elif key == "swin_tiny":
            return timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)

    def _configure_backbone(self):
        # Freeze all first
        for p in self.backbone.parameters(): p.requires_grad = False
        key = self.backbone_key
        btype = self.backbone_type
        if btype == "torchvision":
            if "resnet" in key:
                for p in self.backbone[-1].parameters(): p.requires_grad = True
            elif "densenet" in key:
                for p in self.backbone.features.denseblock4.parameters(): p.requires_grad = True
            elif "efficientnet" in key or "mobilenet" in key:
                total = len(self.backbone.features)
                for i in range(total-5, total-1):
                    for p in self.backbone.features[i].parameters(): p.requires_grad = True
            elif "vgg" in key:
                for p in list(self.backbone.features.parameters())[-10:]: p.requires_grad = True
        else:  # timm
            if   hasattr(self.backbone, "stages"):
                for p in self.backbone.stages[-1].parameters(): p.requires_grad = True
            elif hasattr(self.backbone, "blocks"):
                for p in self.backbone.blocks[-2:].parameters(): p.requires_grad = True
            elif hasattr(self.backbone, "layers"):
                for p in self.backbone.layers[-1].parameters(): p.requires_grad = True
        # Freeze last layer (freeze_last_layer=True in config)
        if   "resnet" in key:
            for p in list(self.backbone[-1].parameters())[-2:]: p.requires_grad = False
        elif "densenet" in key:
            for p in list(self.backbone.features.parameters())[-2:]: p.requires_grad = False
        elif "efficientnet" in key or "mobilenet" in key:
            for p in self.backbone.features[-1].parameters(): p.requires_grad = False
        elif btype == "timm" and hasattr(self.backbone, "stages"):
            for p in list(self.backbone.stages[-1].parameters())[-2:]: p.requires_grad = False

    def unfreeze_for_full_training(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        # Re-freeze last layer
        key = self.backbone_key
        if "resnet" in key:
            for p in list(self.backbone[-1].parameters())[-2:]: p.requires_grad = False
        elif "densenet" in key:
            for p in list(self.backbone.features.parameters())[-2:]: p.requires_grad = False
        elif "efficientnet" in key or "mobilenet" in key:
            for p in self.backbone.features[-1].parameters(): p.requires_grad = False

    def forward(self, x, return_features_only=False):
        f = self.backbone(x)
        if len(f.shape) == 4: f = self.adaptive_pool(f)
        f = f.view(f.size(0), -1)
        f = self.feature_normalizer(f)
        return f if return_features_only else self.temp_classifier(f)

    def get_output_dim(self): return self.backbone_features

print("✓ CBAM + UniversalCNNExtractor ready")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Loss Functions — copied verbatim from training notebook
# ─────────────────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma; self.reduction=reduction
    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce)
        fl = ((1-pt)**self.gamma)*ce
        if self.alpha is not None:
            fl = fl * self.alpha[targets]
        return fl.mean() if self.reduction=="mean" else fl

class LabelSmoothingFocalLoss(nn.Module):
    def __init__(self, num_classes=4, alpha=None, gamma=2.0, smoothing=0.1, reduction="mean"):
        super().__init__()
        self.num_classes=num_classes; self.alpha=alpha
        self.gamma=gamma; self.smoothing=smoothing; self.reduction=reduction
    def forward(self, inputs, targets):
        sl = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
        sl = sl*(1-self.smoothing) + self.smoothing/self.num_classes
        lp = torch.log_softmax(inputs, dim=1)
        loss = (-sl*lp).sum(dim=1)
        pt   = torch.exp(-loss)
        fl   = ((1-pt)**self.gamma)*loss
        if self.alpha is not None:
            fl = fl*self.alpha[targets]
        return fl.mean() if self.reduction=="mean" else fl

def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha>0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam*criterion(pred,ya)+(1-lam)*criterion(pred,yb)

print("✓ Loss functions ready")


## 📐 Shared Metrics & Plotting Utilities

In [ ]:
def compute_metrics(y_true, y_pred, y_probs):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    per_class = {}
    for i, cls in enumerate(CLASS_NAMES):
        tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
        tn=cm.sum()-tp-fp-fn
        sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
        spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        f1   = 2*prec*sens/(prec+sens) if (prec+sens)>0 else 0.0
        per_class[cls] = dict(sensitivity=sens, specificity=spec,
                               precision=prec, f1_score=f1,
                               TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))
    try:
        yb  = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        roc = float(roc_auc_score(yb, y_probs, average="macro", multi_class="ovr"))
    except: roc = float("nan")
    return {
        "accuracy"        : float(np.mean(np.array(y_pred)==np.array(y_true))),
        "macro_f1"        : float(np.mean([m["f1_score"]    for m in per_class.values()])),
        "macro_precision" : float(np.mean([m["precision"]   for m in per_class.values()])),
        "macro_recall"    : float(np.mean([m["sensitivity"] for m in per_class.values()])),
        "roc_auc"         : roc,
        "per_class"       : per_class,
        "confusion_matrix": cm,
    }


def plot_cm(cm, title, out_path, figsize=(7,6)):
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt="d" if cm.dtype==int else ".1f",
                cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, annot_kws={"size":11})
    plt.title(title, fontsize=12, fontweight="bold")
    plt.ylabel("True"); plt.xlabel("Predicted")
    plt.tight_layout(); plt.savefig(out_path, dpi=250, bbox_inches="tight"); plt.close()


def plot_roc(y_true, y_probs, title, out_path):
    yb = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    colours = ["#e74c3c","#3498db","#2ecc71","#9b59b6"]
    macro_fpr = np.linspace(0,1,200); tprs=[]
    fig, (ax1,ax2) = plt.subplots(1,2, figsize=(13,5))
    for i, cls in enumerate(CLASS_NAMES):
        fpr,tpr,_ = roc_curve(yb[:,i], y_probs[:,i])
        a = auc(fpr,tpr)
        ax1.plot(fpr,tpr,color=colours[i],lw=2,label=f"{cls} (AUC={a:.3f})")
        tprs.append(np.interp(macro_fpr,fpr,tpr))
    ax1.plot([0,1],[0,1],"k--",lw=1); ax1.set(xlabel="FPR",ylabel="TPR",title="Per-Class ROC")
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
    mean_tpr=np.mean(tprs,0); ma=auc(macro_fpr,mean_tpr)
    ax2.plot(macro_fpr,mean_tpr,"navy",lw=2.5,label=f"Macro AUC={ma:.4f}")
    ax2.fill_between(macro_fpr,mean_tpr,alpha=0.1,color="navy")
    ax2.plot([0,1],[0,1],"k--",lw=1); ax2.set(xlabel="FPR",ylabel="TPR",title="Macro-avg ROC")
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.close()


def plot_per_class_bars(per_class, title, out_path):
    mets  = ["precision","sensitivity","f1_score","specificity"]
    labs  = ["Precision","Recall","F1","Specificity"]
    cols  = ["#3498db","#2ecc71","#e74c3c","#9b59b6"]
    x=np.arange(len(CLASS_NAMES)); w=0.2
    fig,ax=plt.subplots(figsize=(11,5))
    for j,(m,lab,col) in enumerate(zip(mets,labs,cols)):
        vals=[per_class[c][m] for c in CLASS_NAMES]
        bars=ax.bar(x+(j-1.5)*w, vals, w, label=lab, color=col, alpha=0.85)
        for bar,v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=10)
    ax.set_ylim(0,1.18); ax.set_ylabel("Score")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight"); plt.close()


def save_metrics_csv(metrics, model_name, inf_time, out_dir, suffix=""):
    rows=[["Metric","Value"],["Model",model_name],
          ["Accuracy",   f"{metrics['accuracy']:.4f}"],
          ["Macro F1",   f"{metrics['macro_f1']:.4f}"],
          ["Precision",  f"{metrics['macro_precision']:.4f}"],
          ["Recall",     f"{metrics['macro_recall']:.4f}"],
          ["ROC-AUC",    f"{metrics['roc_auc']:.4f}"],
          ["Infer ms/img",f"{inf_time:.3f}"]]
    for cls in CLASS_NAMES:
        m=metrics["per_class"][cls]
        rows+=[[f"{cls} F1",f"{m['f1_score']:.4f}"],
               [f"{cls} Prec",f"{m['precision']:.4f}"],
               [f"{cls} Rec",f"{m['sensitivity']:.4f}"],
               [f"{cls} Spec",f"{m['specificity']:.4f}"],
               [f"{cls} TP",m["TP"]],[f"{cls} FP",m["FP"]],
               [f"{cls} FN",m["FN"]],[f"{cls} TN",m["TN"]]]
    df=pd.DataFrame(rows[1:], columns=rows[0])
    df.to_csv(os.path.join(out_dir, f"metrics{suffix}.csv"), index=False)


def run_inference(model, loader, label=""):
    model.eval()
    all_true,all_pred,all_probs,times=[],[],[],[]
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc=f"  Infer {label}", leave=False):
            imgs=imgs.to(DEVICE)
            if DEVICE.type=="cuda": torch.cuda.synchronize()
            t0=time.time(); out=model(imgs)
            if DEVICE.type=="cuda": torch.cuda.synchronize()
            times.append((time.time()-t0)*1000/imgs.size(0))
            probs=torch.softmax(out,dim=1).cpu().numpy()
            all_true.extend(lbls.numpy()); all_pred.extend(probs.argmax(1))
            all_probs.extend(probs)
    return (np.array(all_true), np.array(all_pred),
            np.array(all_probs), float(np.mean(times)))


def print_results(metrics, inf_time, model_name):
    print(f"\n{'─'*65}")
    print(f"  {model_name}")
    print(f"{'─'*65}")
    print(f"  Accuracy        : {metrics['accuracy']:.4f}")
    print(f"  Macro Precision : {metrics['macro_precision']:.4f}")
    print(f"  Macro Recall    : {metrics['macro_recall']:.4f}")
    print(f"  Macro F1        : {metrics['macro_f1']:.4f}")
    print(f"  ROC-AUC         : {metrics['roc_auc']:.4f}")
    print(f"  Infer ms/img    : {inf_time:.2f}")
    print(f"{'─'*65}")
    print(f"  {'Class':<14}{'Prec':>8}{'Rec':>8}{'F1':>8}{'Spec':>10}",
          f"{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>5}")
    for cls in CLASS_NAMES:
        m=metrics["per_class"][cls]
        print(f"  {cls:<14}{m['precision']:>8.4f}{m['sensitivity']:>8.4f}"
              f"{m['f1_score']:>8.4f}{m['specificity']:>10.4f}"
              f"{m['TP']:>5}{m['FP']:>5}{m['FN']:>5}{m['TN']:>5}")

print("✓ Metrics & plotting helpers ready")


---
## 🔬 Ablation 1 — DenseNet121 + CBAM + Deep Dense Head + Softmax (No QNN)

**Comparison:**
- **Original:** `feature_extractor` → `CBAM` → `quantum_preprocessing` → `QNN` → `cat(classical, quantum)` → `classifier(backbone+qubits → 512→256→128→64→4)`  
- **A1:**       `feature_extractor` → `CBAM` →  *(QNN removed)*  → `classifier(backbone → 512→256→128→64→4)` → Softmax

The classifier input dimension changes from `backbone_dim + n_qubits` to `backbone_dim` only.  
Everything else (CBAM, feature_normalizer, head depth/width, loss, LR schedule, unfreeze epoch) is identical.


In [ ]:
class A1_DenseNet_NoQNN(nn.Module):
    """
    HybridQNN with quantum branch removed.
    Keeps: UniversalCNNExtractor (DenseNet121) + CBAM + deep classifier head.
    Removes: quantum_weights, quantum_preprocessing, qnode.
    Classifier input: backbone_dim only (vs backbone_dim+n_qubits in original).
    """
    def __init__(self):
        super().__init__()
        self.feature_extractor = UniversalCNNExtractor("densenet121")
        feat_dim = self.feature_extractor.get_output_dim()   # 1024

        self.attention = CBAM(feat_dim, reduction=16)        # same as original

        # Same depth & width as HybridQNN classifier
        # Original: Linear(feat_dim + n_qubits, 512) — here: Linear(feat_dim, 512)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, 256),      nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128),      nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(128, 64),       nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(DROPOUT*0.5),
            nn.Linear(64, NUM_CLASSES)
            # Softmax via loss during training; torch.softmax at inference
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0)

    def forward(self, x, return_features_only=False):
        f = self.feature_extractor(x, return_features_only=True)  # (B, 1024)
        f = self.attention(f)                                       # CBAM
        if return_features_only: return f
        return self.classifier(f)

print("✓ A1 model architecture ready")
print("  DenseNet121 → CBAM → 512→256→128→64→4 + Softmax")
print("  (QNN branch removed, all else identical to training)")


In [ ]:
def train_a1(data_loaders, out_dir, log):
    """
    Two-phase training mirroring original training notebook:
      Phase 1 — fine-tune backbone (PRETRAIN_EPOCHS, CrossEntropy, ReduceLROnPlateau)
      Phase 2 — full training     (NUM_EPOCHS,      LabelSmoothingFocalLoss,
                                    CosineAnnealingWarmRestarts, unfreeze at UNFREEZE_EPOCH)
    """
    os.makedirs(out_dir, exist_ok=True)
    model = A1_DenseNet_NoQNN().to(DEVICE)
    class_weights = data_loaders["train_dataset"].get_class_weights().to(DEVICE)

    # ── Phase 1: Fine-tune (mirrors fine_tune_backbone) ────────────────────
    print("\nPHASE 1: Fine-tuning DenseNet121 (A1)")
    print("="*60)
    criterion = nn.CrossEntropyLoss()
    opt = optim.Adam([
        {"params": model.feature_extractor.backbone.parameters(),
         "lr": PRETRAIN_LR * 0.1},
        {"params": model.feature_extractor.feature_normalizer.parameters(),
         "lr": PRETRAIN_LR},
        {"params": model.feature_extractor.temp_classifier.parameters(),
         "lr": PRETRAIN_LR},
    ], weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    best_val_acc = 0.0

    for epoch in range(PRETRAIN_EPOCHS):
        model.train()
        for imgs, labels in tqdm(data_loaders["train_loader"],
                                  desc=f"  P1 Ep{epoch+1}/{PRETRAIN_EPOCHS}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward(); opt.step()
        # Validation
        model.eval(); correct=total=0
        val_loss=0.0
        with torch.no_grad():
            for imgs,labels in data_loaders["val_loader"]:
                out=model(imgs.to(DEVICE)); tgt=labels.to(DEVICE)
                val_loss+=criterion(out,tgt).item()
                correct+=(out.argmax(1)==tgt).sum().item(); total+=tgt.size(0)
        val_loss/=len(data_loaders["val_loader"])
        val_acc=correct/total; sched.step(val_loss)
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            log.info(f"  A1 P1 Ep{epoch+1}: ValAcc={val_acc:.4f} ★")
            print(f"  Ep{epoch+1:2d}: ValAcc={val_acc:.4f} ★")
    print(f"  Phase 1 complete | Best ValAcc: {best_val_acc:.4f}")

    # ── Phase 2: Full training (mirrors train_hybrid_qnn) ─────────────────
    print("\nPHASE 2: Full training A1 (LabelSmoothingFocalLoss + CosineAnnealing)")
    print("="*60)
    criterion2 = LabelSmoothingFocalLoss(NUM_CLASSES, alpha=class_weights,
                                          gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING)
    opt2 = optim.AdamW([
        {"params": model.feature_extractor.backbone.parameters(),
         "lr": LEARNING_RATE*0.01, "name": "backbone"},
        {"params": list(model.feature_extractor.feature_normalizer.parameters()) +
                   list(model.attention.parameters()) +
                   list(model.classifier.parameters()),
         "lr": LEARNING_RATE, "name": "other"},
    ], weight_decay=WEIGHT_DECAY)
    sched2 = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt2, T_0=COSINE_T0, T_mult=COSINE_TMULT, eta_min=1e-7)

    best_val_acc2=0.0; best_state=None; patience_cnt=0; unfrozen=False

    for epoch in range(NUM_EPOCHS):
        if epoch == UNFREEZE_EPOCH and not unfrozen:
            print(f"  *** UNFREEZING backbone at epoch {epoch+1} ***")
            model.feature_extractor.unfreeze_for_full_training()
            for pg in opt2.param_groups:
                if pg.get("name")=="backbone": pg["lr"]=LEARNING_RATE*0.01
            unfrozen=True

        model.train()
        for imgs, labels in tqdm(data_loaders["train_loader"],
                                  desc=f"  P2 Ep{epoch+1}/{NUM_EPOCHS}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt2.zero_grad()
            if USE_MIXUP and np.random.random()>0.5:
                mx, ya, yb, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                loss = mixup_criterion(criterion2, model(mx), ya, yb, lam)
            else:
                loss = criterion2(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()
        sched2.step()

        model.eval(); correct=total=0; val_loss2=0.0
        with torch.no_grad():
            for imgs,labels in data_loaders["val_loader"]:
                out=model(imgs.to(DEVICE)); tgt=labels.to(DEVICE)
                val_loss2+=criterion2(out,tgt).item()
                correct+=(out.argmax(1)==tgt).sum().item(); total+=tgt.size(0)
        val_loss2/=len(data_loaders["val_loader"]); val_acc2=correct/total
        if val_acc2>best_val_acc2:
            best_val_acc2=val_acc2; best_state=copy.deepcopy(model.state_dict())
            patience_cnt=0
            log.info(f"  A1 P2 Ep{epoch+1}: ValAcc={val_acc2:.4f} ★")
            print(f"  Ep{epoch+1:3d}: ValAcc={val_acc2:.4f} ★")
        else:
            patience_cnt+=1
            if patience_cnt>=PATIENCE:
                print(f"  Early stop at epoch {epoch+1}"); break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(out_dir, "a1_model.pth"))

    # ── Evaluate ──────────────────────────────────────────────────────────
    y_true, y_pred, y_probs, inf_t = run_inference(model, data_loaders["test_loader"], "A1")
    metrics = compute_metrics(y_true, y_pred, y_probs)
    metrics.update({"inf_time_ms":inf_t,"y_true":y_true,"y_pred":y_pred,"y_probs":y_probs})

    print_results(metrics, inf_t, "A1 — DenseNet121+CBAM+DenseHead+Softmax")
    log.info(f"A1: Acc={metrics['accuracy']:.4f} F1={metrics['macro_f1']:.4f} "
             f"AUC={metrics['roc_auc']:.4f} Infer={inf_t:.2f}ms")

    save_metrics_csv(metrics, "A1_DenseNet_Softmax", inf_t, out_dir)
    plot_cm(metrics["confusion_matrix"], "A1 — DenseNet121+CBAM+DenseHead+Softmax",
            os.path.join(out_dir, "confusion_matrix.png"))
    plot_roc(y_true, y_probs, "A1 — ROC Curves",
             os.path.join(out_dir, "roc_curves.png"))
    plot_per_class_bars(metrics["per_class"], "A1 — Per-Class Metrics",
                        os.path.join(out_dir, "per_class_bars.png"))

    print(f"\n✅ A1 complete — results saved to {out_dir}/")
    return metrics

print("✓ A1 training function ready")


In [ ]:
# ── RUN ABLATION 1 ────────────────────────────────────────────────────────────
print("=" * 70)
print("  ABLATION 1: DenseNet121 + CBAM + Deep Dense Head + Softmax (No QNN)")
print("=" * 70)
log.info("="*60); log.info("ABLATION 1: DenseNet121+CBAM+DenseHead+Softmax")

a1_dir = os.path.join(OUTPUT_DIR, "A1_DenseNet_Softmax")
a1_metrics = train_a1(data_loaders, a1_dir, log)

if torch.cuda.is_available(): torch.cuda.empty_cache()


---
## 🔬 Ablation 2 — DenseNet121 + CBAM Features → Classical ML Classifiers

**What changes:** QNN *and* deep neural classifier are both removed.  
**Feature extraction:** DenseNet121 + CBAM (frozen, ImageNet weights) → 1024-D feature vector.  
**Classifiers:** SVM (RBF), SVM (Linear), Random Forest, Gradient Boosting, Logistic Regression, KNN.  
Features are extracted once; each classifier is fit independently on the same feature set.


In [ ]:
class A2_FeatureExtractor(nn.Module):
    """Frozen DenseNet121 + CBAM for feature extraction (no fine-tuning)."""
    def __init__(self):
        super().__init__()
        self.feature_extractor = UniversalCNNExtractor("densenet121")
        self.attention          = CBAM(self.feature_extractor.get_output_dim(), reduction=16)
        # Freeze all — pure feature extraction with ImageNet weights
        for p in self.parameters(): p.requires_grad = False

    def forward(self, x):
        f = self.feature_extractor(x, return_features_only=True)
        return self.attention(f)   # (B, 1024)


A2_CLASSIFIERS = {
    "SVM_RBF"           : Pipeline([("sc", StandardScaler()),
                                     ("clf", SVC(kernel="rbf", C=10, gamma="scale",
                                                  probability=True, random_state=42))]),
    "SVM_Linear"        : Pipeline([("sc", StandardScaler()),
                                     ("clf", SVC(kernel="linear", C=1,
                                                  probability=True, random_state=42))]),
    "RandomForest"      : RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),

    "KNN"               : Pipeline([("sc", StandardScaler()),
                                     ("clf", KNeighborsClassifier(n_neighbors=5, n_jobs=-1))]),
}


def extract_features(extractor, loader):
    extractor.eval()
    X, y = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="  Extracting features", leave=False):
            f = extractor(imgs.to(DEVICE)).cpu().numpy()
            X.extend(f); y.extend(lbls.numpy())
    return np.array(X), np.array(y)

print("✓ A2 feature extractor and classifiers defined")
print(f"  Feature dim : {BACKBONE_FEATURES['densenet121']}  ({len(A2_CLASSIFIERS)} classifiers)")


In [ ]:
def run_a2(data_loaders, out_dir, log):
    """Extract features once, fit & evaluate all classical ML classifiers."""
    os.makedirs(out_dir, exist_ok=True)

    extractor = A2_FeatureExtractor().to(DEVICE)

    print("  Extracting train features...")
    X_train, y_train = extract_features(extractor, data_loaders["train_loader"])
    print("  Extracting test  features...")
    X_test,  y_test  = extract_features(extractor, data_loaders["test_loader"])
    print(f"  Train: {X_train.shape}  Test: {X_test.shape}")

    all_metrics = {}
    summary_rows = []

    for clf_name, clf in A2_CLASSIFIERS.items():
        print(f"\n  ─── {clf_name} ───")
        clf_copy = copy.deepcopy(clf)
        t0 = time.time()
        clf_copy.fit(X_train, y_train)
        fit_time = time.time()-t0

        t_inf0 = time.time()
        y_pred = clf_copy.predict(X_test)
        inf_ms = (time.time()-t_inf0)*1000/max(len(X_test),1)

        try:    y_probs = clf_copy.predict_proba(X_test)
        except: y_probs = np.eye(NUM_CLASSES)[y_pred]

        metrics = compute_metrics(y_test, y_pred, y_probs)
        metrics["inf_time_ms"] = inf_ms
        all_metrics[clf_name]  = metrics

        print_results(metrics, inf_ms, f"A2 — {clf_name}")
        log.info(f"A2 {clf_name}: Acc={metrics['accuracy']:.4f} "
                 f"F1={metrics['macro_f1']:.4f} AUC={metrics['roc_auc']:.4f} "
                 f"FitTime={fit_time:.1f}s Infer={inf_ms:.2f}ms")

        clf_dir = os.path.join(out_dir, clf_name)
        os.makedirs(clf_dir, exist_ok=True)
        save_metrics_csv(metrics, f"A2_{clf_name}", inf_ms, clf_dir)
        plot_cm(metrics["confusion_matrix"],
                f"A2 {clf_name} — Confusion Matrix",
                os.path.join(clf_dir, "confusion_matrix.png"))
        plot_roc(y_test, y_probs, f"A2 {clf_name} — ROC",
                 os.path.join(clf_dir, "roc_curves.png"))
        plot_per_class_bars(metrics["per_class"], f"A2 {clf_name} — Per-Class",
                            os.path.join(clf_dir, "per_class_bars.png"))

        summary_rows.append({
            "Classifier"  : clf_name,
            "Accuracy"    : f"{metrics['accuracy']:.4f}",
            "Macro F1"    : f"{metrics['macro_f1']:.4f}",
            "Precision"   : f"{metrics['macro_precision']:.4f}",
            "Recall"      : f"{metrics['macro_recall']:.4f}",
            "ROC-AUC"     : f"{metrics['roc_auc']:.4f}",
            "Infer ms/img": f"{inf_ms:.2f}",
            "Fit Time s"  : f"{fit_time:.1f}",
        })

    # Summary table
    df = pd.DataFrame(summary_rows).sort_values("Accuracy", ascending=False)
    df.to_csv(os.path.join(out_dir, "all_classifiers_summary.csv"), index=False)
    print(f"\n{'='*70}\n  A2 — Classifier Summary\n{'='*70}")
    print(df.to_string(index=False))
    log.info(f"A2 Summary:\n{df.to_string(index=False)}")

    # Best classifier CM highlight
    best = df.iloc[0]["Classifier"]
    print(f"\n  Best A2 classifier: {best}")
    print(f"\n✅ A2 complete — results saved to {out_dir}/")
    return all_metrics, df

print("✓ A2 training function ready")


In [ ]:
# ── RUN ABLATION 2 ────────────────────────────────────────────────────────────
print("=" * 70)
print("  ABLATION 2: DenseNet121+CBAM Features → Classical ML Classifiers")
print("=" * 70)
log.info("="*60); log.info("ABLATION 2: Classical ML classifiers")

a2_dir = os.path.join(OUTPUT_DIR, "A2_Classical_ML")
a2_all_metrics, a2_summary_df = run_a2(data_loaders, a2_dir, log)

if torch.cuda.is_available(): torch.cuda.empty_cache()


---
## 🔬 Ablation 3 — 8 Backbones × Standard Classifier Head + Softmax (No QNN, No CBAM)

**What changes:** QNN, CBAM, and the custom deep dense classifier are all removed.  
**What remains:** Each backbone + its own single linear head (`Linear(features → 4)`) + Softmax.  
This is the standard transfer-learning baseline — purely evaluating whether the QNN+CBAM additions provide value.  
**Training:** Same two-phase approach (Phase 1 = fine-tune backbone, Phase 2 = full training with LabelSmoothingFocalLoss + CosineAnnealing + unfreeze at epoch 15).


In [ ]:
def build_a3_standard_model(backbone_key):
    """
    Standard backbone + single Linear(features→4) head.
    No CBAM, no QNN, no deep classifier stack.
    Pretrained ImageNet weights for all backbones.
    """
    key = backbone_key
    if key == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(512, NUM_CLASSES)
    elif key == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(2048, NUM_CLASSES)
    elif key == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(1024, NUM_CLASSES)
    elif key == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, NUM_CLASSES))
    elif key == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, NUM_CLASSES))
    elif key == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[-1] = nn.Linear(4096, NUM_CLASSES)
    elif key == "convnext_tiny":
        m = timm.create_model("convnext_tiny", pretrained=True, num_classes=NUM_CLASSES)
    elif key == "swin_tiny":
        m = timm.create_model("swin_tiny_patch4_window7_224",
                               pretrained=True, num_classes=NUM_CLASSES)
    return m


def get_a3_param_groups(model, backbone_key, lr):
    """Separate backbone vs head parameter groups (mirrors fine_tune_backbone)."""
    head_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if any(k in name for k in ["fc","classifier","head","norm"]):
            head_params.append(p)
        else:
            backbone_params.append(p)
    return [
        {"params": backbone_params, "lr": lr*0.01, "name": "backbone"},
        {"params": head_params,     "lr": lr,       "name": "head"},
    ]

print("✓ A3 standard backbone builders ready")


In [ ]:
def train_a3_backbone(backbone_key, data_loaders, out_dir, log):
    """
    Two-phase training identical to original for a standard backbone+head.
    Phase 1: fine-tune head (CrossEntropy, ReduceLROnPlateau, PRETRAIN_EPOCHS)
    Phase 2: full training  (LabelSmoothingFocalLoss, CosineAnnealing, NUM_EPOCHS)
    """
    os.makedirs(out_dir, exist_ok=True)
    model = build_a3_standard_model(backbone_key).to(DEVICE)
    class_weights = data_loaders["train_dataset"].get_class_weights().to(DEVICE)

    # ── Phase 1: Fine-tune ────────────────────────────────────────────────
    print(f"\n  P1: Fine-tuning {backbone_key}")
    # Freeze backbone, only train head
    for name, p in model.named_parameters():
        if not any(k in name for k in ["fc","classifier","head"]):
            p.requires_grad = False
    criterion = nn.CrossEntropyLoss()
    opt = optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    best_p1_acc = 0.0

    for epoch in range(PRETRAIN_EPOCHS):
        model.train()
        for imgs, labels in tqdm(data_loaders["train_loader"],
                                  desc=f"  P1 Ep{epoch+1}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(); loss=criterion(model(imgs),labels)
            loss.backward(); opt.step()
        model.eval(); correct=total=0; vl=0.0
        with torch.no_grad():
            for imgs,labels in data_loaders["val_loader"]:
                out=model(imgs.to(DEVICE)); tgt=labels.to(DEVICE)
                vl+=criterion(out,tgt).item()
                correct+=(out.argmax(1)==tgt).sum().item(); total+=tgt.size(0)
        vl/=len(data_loaders["val_loader"]); va=correct/total; sched.step(vl)
        if va>best_p1_acc:
            best_p1_acc=va; print(f"  Ep{epoch+1:2d}: ValAcc={va:.4f} ★")

    # ── Phase 2: Full training ─────────────────────────────────────────────
    print(f"  P2: Full training {backbone_key}")
    for p in model.parameters(): p.requires_grad = True  # Unfreeze all initially
    criterion2 = LabelSmoothingFocalLoss(NUM_CLASSES, alpha=class_weights,
                                          gamma=FOCAL_GAMMA, smoothing=LABEL_SMOOTHING)
    opt2 = optim.AdamW(get_a3_param_groups(model, backbone_key, LEARNING_RATE),
                        weight_decay=WEIGHT_DECAY)
    sched2 = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt2, T_0=COSINE_T0, T_mult=COSINE_TMULT, eta_min=1e-7)

    best_val_acc=0.0; best_state=None; patience_cnt=0

    for epoch in range(NUM_EPOCHS):
        model.train()
        for imgs, labels in tqdm(data_loaders["train_loader"],
                                  desc=f"  P2 Ep{epoch+1}", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt2.zero_grad()
            if USE_MIXUP and np.random.random()>0.5:
                mx, ya, yb, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                loss = mixup_criterion(criterion2, model(mx), ya, yb, lam)
            else:
                loss = criterion2(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt2.step()
        sched2.step()

        model.eval(); correct=total=0
        with torch.no_grad():
            for imgs,labels in data_loaders["val_loader"]:
                out=model(imgs.to(DEVICE)); tgt=labels.to(DEVICE)
                correct+=(out.argmax(1)==tgt).sum().item(); total+=tgt.size(0)
        val_acc=correct/total
        if val_acc>best_val_acc:
            best_val_acc=val_acc; best_state=copy.deepcopy(model.state_dict())
            patience_cnt=0
            log.info(f"  A3 {backbone_key} P2 Ep{epoch+1}: ValAcc={val_acc:.4f} ★")
            print(f"  Ep{epoch+1:3d}: ValAcc={val_acc:.4f} ★")
        else:
            patience_cnt+=1
            if patience_cnt>=PATIENCE:
                print(f"  Early stop ep {epoch+1}"); break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(out_dir, f"{backbone_key}_model.pth"))

    # ── Evaluate ──────────────────────────────────────────────────────────
    y_true, y_pred, y_probs, inf_t = run_inference(
        model, data_loaders["test_loader"], f"A3-{backbone_key}")
    metrics = compute_metrics(y_true, y_pred, y_probs)
    metrics.update({"inf_time_ms":inf_t,"y_true":y_true,"y_pred":y_pred,"y_probs":y_probs})

    print_results(metrics, inf_t, f"A3 — {backbone_key} + Standard Head + Softmax")
    log.info(f"A3 {backbone_key}: Acc={metrics['accuracy']:.4f} "
             f"F1={metrics['macro_f1']:.4f} AUC={metrics['roc_auc']:.4f} "
             f"Infer={inf_t:.2f}ms")

    save_metrics_csv(metrics, f"A3_{backbone_key}", inf_t, out_dir)
    plot_cm(metrics["confusion_matrix"],
            f"A3 {backbone_key} — Standard Head + Softmax",
            os.path.join(out_dir, "confusion_matrix.png"))
    plot_roc(y_true, y_probs, f"A3 {backbone_key} — ROC",
             os.path.join(out_dir, "roc_curves.png"))
    plot_per_class_bars(metrics["per_class"], f"A3 {backbone_key} — Per-Class",
                        os.path.join(out_dir, "per_class_bars.png"))

    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return metrics

print("✓ A3 training function ready")


In [ ]:
# ── RUN ABLATION 3 ────────────────────────────────────────────────────────────
print("=" * 70)
print("  ABLATION 3: 8 Backbones × Standard Classifier + Softmax (No QNN)")
print("=" * 70)
log.info("="*60); log.info("ABLATION 3: 8 Backbones Standard Classifier")

a3_dir = os.path.join(OUTPUT_DIR, "A3_Standard_Backbones")
os.makedirs(a3_dir, exist_ok=True)

BACKBONES_A3 = list(BACKBONE_FEATURES.keys())
a3_all = {}

for bkey in BACKBONES_A3:
    print(f"\n{'─'*70}")
    print(f"  Backbone : {bkey.upper()}")
    print(f"{'─'*70}")
    bdir = os.path.join(a3_dir, bkey)
    a3_all[bkey] = train_a3_backbone(bkey, data_loaders, bdir, log)

# Summary table
a3_rows = []
for bkey, m in a3_all.items():
    a3_rows.append({
        "Backbone"    : bkey,
        "Accuracy"    : f"{m['accuracy']:.4f}",
        "Macro F1"    : f"{m['macro_f1']:.4f}",
        "Precision"   : f"{m['macro_precision']:.4f}",
        "Recall"      : f"{m['macro_recall']:.4f}",
        "ROC-AUC"     : f"{m['roc_auc']:.4f}",
        "Infer ms/img": f"{m['inf_time_ms']:.2f}",
    })
a3_df = pd.DataFrame(a3_rows).sort_values("Accuracy", ascending=False)
a3_df.to_csv(os.path.join(a3_dir, "backbone_comparison.csv"), index=False)
print(f"\n{'='*70}\n  A3 — Backbone Comparison\n{'='*70}")
print(a3_df.to_string(index=False))
log.info(f"A3 Backbone comparison:\n{a3_df.to_string(index=False)}")
print("\n✅ Ablation 3 complete")


---
## 📊 Final Ablation Comparison — All Variants

In [ ]:
# ── Master comparison table ───────────────────────────────────────────────────
print("=" * 70)
print("  MASTER ABLATION COMPARISON")
print("=" * 70)

master_rows = []

# A1
master_rows.append({
    "Ablation"    : "A1",
    "Config"      : "DenseNet121+CBAM+DenseHead+Softmax",
    "Accuracy"    : f"{a1_metrics['accuracy']:.4f}",
    "Macro F1"    : f"{a1_metrics['macro_f1']:.4f}",
    "Precision"   : f"{a1_metrics['macro_precision']:.4f}",
    "Recall"      : f"{a1_metrics['macro_recall']:.4f}",
    "ROC-AUC"     : f"{a1_metrics['roc_auc']:.4f}",
    "Infer ms/img": f"{a1_metrics['inf_time_ms']:.2f}",
})

# A2
for clf_name, m in a2_all_metrics.items():
    master_rows.append({
        "Ablation"    : "A2",
        "Config"      : f"DenseNet121+CBAM → {clf_name}",
        "Accuracy"    : f"{m['accuracy']:.4f}",
        "Macro F1"    : f"{m['macro_f1']:.4f}",
        "Precision"   : f"{m['macro_precision']:.4f}",
        "Recall"      : f"{m['macro_recall']:.4f}",
        "ROC-AUC"     : f"{m['roc_auc']:.4f}",
        "Infer ms/img": f"{m['inf_time_ms']:.2f}",
    })

# A3
for bkey, m in a3_all.items():
    master_rows.append({
        "Ablation"    : "A3",
        "Config"      : f"{bkey}+StandardHead+Softmax",
        "Accuracy"    : f"{m['accuracy']:.4f}",
        "Macro F1"    : f"{m['macro_f1']:.4f}",
        "Precision"   : f"{m['macro_precision']:.4f}",
        "Recall"      : f"{m['macro_recall']:.4f}",
        "ROC-AUC"     : f"{m['roc_auc']:.4f}",
        "Infer ms/img": f"{m['inf_time_ms']:.2f}",
    })

master_df = pd.DataFrame(master_rows)
master_df.to_csv(os.path.join(OUTPUT_DIR, "MASTER_ABLATION_SUMMARY.csv"), index=False)
print(master_df.to_string(index=False))
log.info(f"MASTER SUMMARY:\n{master_df.to_string(index=False)}")


In [ ]:
# ── Figure 1: Accuracy bar chart across all ablations ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Ablation Study — Test Accuracy Across All Variants",
             fontsize=14, fontweight="bold")

# A1
ax = axes[0]
bar = ax.bar(["DenseNet+CBAM\nDenseHead+Softmax"],
             [a1_metrics["accuracy"]], color="#3498db", alpha=0.85, width=0.5)
ax.text(bar[0].get_x()+bar[0].get_width()/2, a1_metrics["accuracy"]+0.01,
        f"{a1_metrics['accuracy']:.4f}", ha="center", fontsize=11, fontweight="bold")
ax.set_ylim(0,1.15); ax.set_title("A1 — DenseNet121+CBAM\n+DenseHead (No QNN)",
                                    fontsize=10, fontweight="bold")
ax.set_ylabel("Test Accuracy"); ax.grid(axis="y", alpha=0.3)

# A2
ax = axes[1]
clf_names = list(a2_all_metrics.keys())
clf_accs  = [a2_all_metrics[k]["accuracy"] for k in clf_names]
cols2 = ["#e74c3c","#e67e22","#2ecc71","#9b59b6","#1abc9c","#f39c12"]
bars = ax.bar(clf_names, clf_accs, color=cols2[:len(clf_names)], alpha=0.85)
for bar,acc in zip(bars,clf_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{acc:.3f}", ha="center", fontsize=8.5)
ax.set_ylim(0,1.15); ax.set_title("A2 — DenseNet121+CBAM\nFeatures → Classical ML",
                                    fontsize=10, fontweight="bold")
ax.set_xticklabels(clf_names, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Test Accuracy"); ax.grid(axis="y", alpha=0.3)

# A3
ax = axes[2]
b_names = BACKBONES_A3
b_accs  = [a3_all[k]["accuracy"] for k in b_names]
cols3 = ["#3498db","#2ecc71","#e74c3c","#9b59b6","#f39c12","#1abc9c","#e67e22","#e91e63"]
bars = ax.bar(b_names, b_accs, color=cols3, alpha=0.85)
for bar,acc in zip(bars,b_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{acc:.3f}", ha="center", fontsize=8)
ax.set_ylim(0,1.15); ax.set_title("A3 — Standard Backbone\n+Linear Head (No QNN/CBAM)",
                                    fontsize=10, fontweight="bold")
ax.set_xticklabels(b_names, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Test Accuracy"); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_accuracy_comparison.png"),
            dpi=300, bbox_inches="tight"); plt.show()
print("✓ Accuracy comparison chart saved")


In [ ]:
# ── Figure 2: All-metrics heatmap ─────────────────────────────────────────────
metric_keys   = ["accuracy","macro_f1","macro_precision","macro_recall","roc_auc"]
metric_labels = ["Accuracy","Macro F1","Precision","Recall","ROC-AUC"]

config_labels, matrix = [], []

config_labels.append("A1: DenseNet+CBAM+DenseHead")
matrix.append([a1_metrics[mk] for mk in metric_keys])

for clf in A2_CLASSIFIERS:
    config_labels.append(f"A2: {clf}")
    matrix.append([a2_all_metrics[clf][mk] for mk in metric_keys])

for bkey in BACKBONES_A3:
    config_labels.append(f"A3: {bkey}")
    matrix.append([a3_all[bkey][mk] for mk in metric_keys])

mat = np.array(matrix)
fig, ax = plt.subplots(figsize=(12, max(8, len(config_labels)*0.48)))
im = ax.imshow(mat, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(metric_labels))); ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_yticks(range(len(config_labels))); ax.set_yticklabels(config_labels, fontsize=9)
for i in range(len(config_labels)):
    for j in range(len(metric_labels)):
        v = mat[i,j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8,
                color="white" if v>0.7 else "black", fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8, label="Score")
ax.set_title("Ablation Study — All Metrics Heatmap",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_heatmap.png"),
            dpi=300, bbox_inches="tight"); plt.show()
print("✓ Heatmap saved")


In [ ]:
# ── Figure 3: Per-class F1 across all ablations ───────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Per-Class F1 Score — Ablation Study", fontsize=13, fontweight="bold")
colours_cls = ["#e74c3c","#3498db","#2ecc71","#9b59b6"]

for ci, (cls, ax) in enumerate(zip(CLASS_NAMES, axes.ravel())):
    cfgs, f1s = [], []
    cfgs.append("A1"); f1s.append(a1_metrics["per_class"][cls]["f1_score"])
    for clf in A2_CLASSIFIERS:
        cfgs.append(f"A2\n{clf}"); f1s.append(a2_all_metrics[clf]["per_class"][cls]["f1_score"])
    for bkey in BACKBONES_A3:
        cfgs.append(f"A3\n{bkey}"); f1s.append(a3_all[bkey]["per_class"][cls]["f1_score"])
    x=np.arange(len(cfgs))
    bars=ax.bar(x, f1s, color=colours_cls[ci], alpha=0.8, width=0.7)
    ax.set_ylim(0,1.2); ax.set_ylabel("F1 Score")
    ax.set_title(f"Class: {cls}", fontsize=11, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(cfgs, fontsize=7, rotation=30, ha="right")
    ax.grid(axis="y", alpha=0.3)
    for bar,v in zip(bars,f1s):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{v:.3f}", ha="center", fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ablation_perclass_f1.png"),
            dpi=300, bbox_inches="tight"); plt.show()
print("✓ Per-class F1 chart saved")


In [ ]:
# ── Final summary log ─────────────────────────────────────────────────────────
log.info("="*60); log.info("ABLATION STUDY COMPLETE")
log.info(f"A1 DenseNet+CBAM+DenseHead+Softmax: "
         f"Acc={a1_metrics['accuracy']:.4f} F1={a1_metrics['macro_f1']:.4f} "
         f"AUC={a1_metrics['roc_auc']:.4f}")
for clf in A2_CLASSIFIERS:
    m=a2_all_metrics[clf]
    log.info(f"A2 {clf}: Acc={m['accuracy']:.4f} F1={m['macro_f1']:.4f} "
             f"AUC={m['roc_auc']:.4f}")
for bkey in BACKBONES_A3:
    m=a3_all[bkey]
    log.info(f"A3 {bkey}: Acc={m['accuracy']:.4f} F1={m['macro_f1']:.4f} "
             f"AUC={m['roc_auc']:.4f}")
log.info("="*60)

print("="*70)
print("  ALL ABLATION STUDIES COMPLETE")
print(f"  Results : {OUTPUT_DIR}/")
print(f"  Log     : {log_path}")
print("="*70)
print("\nOutput tree:")
print(f"  {OUTPUT_DIR}/")
print(f"  ├── MASTER_ABLATION_SUMMARY.csv")
print(f"  ├── ablation_study.log")
print(f"  ├── ablation_accuracy_comparison.png")
print(f"  ├── ablation_heatmap.png")
print(f"  ├── ablation_perclass_f1.png")
print(f"  ├── A1_DenseNet_Softmax/")
print(f"  │     metrics.csv, confusion_matrix.png, roc_curves.png,")
print(f"  │     per_class_bars.png, a1_model.pth")
print(f"  ├── A2_Classical_ML/")
print(f"  │     all_classifiers_summary.csv")
print(f"  │     {{SVM_RBF, SVM_Linear, RandomForest, ...}}/")
print(f"  │       metrics.csv, confusion_matrix.png, roc_curves.png")
print(f"  └── A3_Standard_Backbones/")
print(f"        backbone_comparison.csv")
print(f"        {{densenet121, resnet18, ...}}/")
print(f"          metrics.csv, confusion_matrix.png, roc_curves.png, model.pth")


# Task
I will create a zip file of the `OUTPUT_DIR` (`ablation_study_results`), then upload this zip file to your Google Drive to `/content/drive/MyDrive/ablation_study_results.zip`.

## Zip results folder

### Subtask:
Create a zip archive of the `OUTPUT_DIR` (ablation_study_results).


**Reasoning**:
To create a zip archive of the specified output directory, I will use the `zip` command-line utility as instructed. The `-r` flag will ensure that the directory and its contents are included recursively, and the output file will be named `ablation_study_results.zip`.



In [ ]:
import shutil
import os

# Ensure OUTPUT_DIR is defined, though it should be from previous cells
if 'OUTPUT_DIR' not in locals():
    OUTPUT_DIR = 'ablation_study_results'

zip_filename = f"{OUTPUT_DIR}.zip"

# Create the zip archive
# shutil.make_archive(base_name, format, root_dir=None, base_dir=None, verbose=0, dry_run=0, logger=None)
# This will create a zip file named OUTPUT_DIR.zip in the current directory, containing the contents of OUTPUT_DIR
shutil.make_archive(OUTPUT_DIR, 'zip', base_dir=OUTPUT_DIR)

print(f"✓ Zip archive created: {zip_filename}")
print(f"  Contents of {zip_filename} are from {os.path.abspath(OUTPUT_DIR)}")

**Reasoning**:
Now that the zip archive has been created, I will upload it to the specified Google Drive path using the `shutil.move` command, moving the file from the current directory to the mounted Google Drive directory.



In [ ]:
import shutil
import os

# Define the destination path in Google Drive
drive_upload_path = "/content/drive/MyDrive/ablation_study_results.zip"

# Move the created zip file to Google Drive
shutil.move(zip_filename, drive_upload_path)

print(f"✓ Zip file '{zip_filename}' uploaded to '{drive_upload_path}'")

## Summary:

### Data Analysis Key Findings

*   A zip archive named `ablation_study_results.zip` was successfully created from the `ablation_study_results` directory.
*   This zip file was then successfully uploaded to Google Drive at `/content/drive/MyDrive/ablation_study_results.zip`.

### Insights or Next Steps

*   The results of the ablation study are now securely stored and accessible in Google Drive.
